In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model
import h5py
import matplotlib.gridspec as gridspec
from scipy.ndimage import zoom
from scipy import interpolate
import os

print(f"TensorFlow Version: {tf.__version__}")
print(f"GPUs Available: {len(tf.config.list_physical_devices('GPU'))}")

# Configuration
LR_DIM = 10
STAGE_DIMS = [20, 40, 80, 200, 400]
OTHER_DETAILS = "progressive_residual_unet_(20-40-80-200-400)_trained_with_LDCs-and-one-bfs-with_aspect_ratio_correction"

# Base Path
BASE_PATH = "/kaggle/input/datasets/amirmohdk/unet-progressive-trained-ldc-and-one-bfs-outputs/"

# Files
STATS_FILE = f"{BASE_PATH}norm_stats_10to400_{OTHER_DETAILS}.txt"
MODEL_FILES = {
    20: f"{BASE_PATH}unet_stage_10to20_{OTHER_DETAILS}.h5",
    40: f"{BASE_PATH}unet_stage_20to40_{OTHER_DETAILS}.h5",
    80: f"{BASE_PATH}unet_stage_40to80_{OTHER_DETAILS}.h5",
    200: f"{BASE_PATH}unet_stage_80to200_{OTHER_DETAILS}.h5",
    400: f"{BASE_PATH}unet_stage_200to400_{OTHER_DETAILS}.h5"
}

In [ ]:
# =========================
# HELPER FUNCTIONS
# =========================

def bicubic_interpolate_batch(x, target_size):
    """
    Bicubic interpolation for batched multi-channel images.
    """
    return tf.image.resize(x, target_size, method='bicubic')

def normalize_with_stats(arr, stats):
    """Apply pre-computed normalization statistics."""
    normalized = np.zeros_like(arr)
    for ch_idx, ch_name in enumerate(['u', 'v', 'p']):
        mean, std = stats[ch_name]
        normalized[..., ch_idx] = (arr[..., ch_idx] - mean) / std
    return normalized

def denormalize_with_stats(arr, stats):
    """Inverse normalization."""
    denormalized = np.zeros_like(arr)
    for ch_idx, ch_name in enumerate(['u', 'v', 'p']):
        mean, std = stats[ch_name]
        denormalized[..., ch_idx] = arr[..., ch_idx] * std + mean
    return denormalized

def reshape_rectangular_to_square(fields_dict, nx, ny, lx, ly):
    """
    Resample rectangular grid data to square coordinate system for ML model.
    Uses bicubic interpolation to map physical domain to canonical square.
    """
    print(f"  🔄 Aspect ratio correction: ({nx}×{ny}) domain [{lx}×{ly}] → square [{lx}×{lx}]")
    
    # Create physical coordinate systems
    x_rect = np.linspace(0, lx, nx)
    y_rect = np.linspace(0, ly, ny)
    
    # Square coordinate system (use max dimension for isotropy)
    L_square = max(lx, ly)
    x_square = np.linspace(0, L_square, nx)
    y_square = np.linspace(0, L_square, nx)
    
    # Resample each field from rectangular to square
    from scipy import interpolate
    fields_square = {}
    for component in ['u', 'v', 'p']:
        # Use RectBivariateSpline for smooth interpolation
        f_interp = interpolate.RectBivariateSpline(y_rect, x_rect, fields_dict[component])
        fields_square[component] = f_interp(y_square, x_square)
    
    return fields_square

def plot_comparison(input_lr, target_hr, pred_hr, stats, title_prefix=""):
    """
    Plot comparison of low-res input, ground truth, and prediction.
    Shows all 3 channels (u, v, p) in separate rows.
    """
    # Denormalize (assuming single sample batch dim)
    if len(target_hr.shape) == 4:
        target_real = denormalize_with_stats(target_hr, stats)[0]
        pred_real = denormalize_with_stats(pred_hr, stats)[0]
        input_real = denormalize_with_stats(input_lr, stats)[0]
    else:
        target_real = denormalize_with_stats(target_hr[None,...], stats)[0]
        pred_real = denormalize_with_stats(pred_hr[None,...], stats)[0]
        input_real = denormalize_with_stats(input_lr[None,...], stats)[0]
    
    fig, axes = plt.subplots(3, 4, figsize=(18, 12))
    channel_names = ['u-velocity', 'v-velocity', 'Pressure']
    cmap = 'RdBu'
    
    for ch_idx in range(3):
        # Input (interpolated for visualization)
        input_interp = bicubic_interpolate_batch(
            input_real[..., ch_idx:ch_idx+1][None, ...], 
            target_real.shape[:2]
        )[0, ..., 0].numpy()
        
        # Extract 2D data for each plot
        # Apply transpose (.T) to match correct geometry orientation
        data_input = input_interp.T
        data_target = target_real[..., ch_idx].T
        data_pred = pred_real[..., ch_idx].T
        diff = data_target - data_pred
        diff_max_abs = np.abs(diff).max()
        if diff_max_abs == 0: diff_max_abs = 1e-8
        mae = np.mean(np.abs(diff))
        
        # Use contourf with levels=20
        im0 = axes[ch_idx, 0].contourf(data_input, levels=20, cmap=cmap)
        fig.colorbar(im0, ax=axes[ch_idx, 0]).set_label("Field Value")
        axes[ch_idx, 0].set_title(f'Input (Interpolated)\n{channel_names[ch_idx]}')
        axes[ch_idx, 0].set_aspect('equal')
        
        im1 = axes[ch_idx, 1].contourf(data_target, levels=20, cmap=cmap)
        fig.colorbar(im1, ax=axes[ch_idx, 1]).set_label("Field Value")
        axes[ch_idx, 1].set_title(f'Ground Truth\n{channel_names[ch_idx]}')
        axes[ch_idx, 1].set_aspect('equal')
        
        im2 = axes[ch_idx, 2].contourf(data_pred, levels=20, cmap=cmap)
        fig.colorbar(im2, ax=axes[ch_idx, 2]).set_label("Field Value")
        axes[ch_idx, 2].set_title(f'Prediction\n{channel_names[ch_idx]}')
        axes[ch_idx, 2].set_aspect('equal')
        
        im3 = axes[ch_idx, 3].contourf(diff, levels=20, cmap=cmap, vmin=-diff_max_abs, vmax=diff_max_abs)
        fig.colorbar(im3, ax=axes[ch_idx, 3]).set_label("Error")
        axes[ch_idx, 3].set_title(f'Error | MAE: {mae:.6f}\n{channel_names[ch_idx]}')
        axes[ch_idx, 3].set_aspect('equal')
    
    fig.suptitle(f'{title_prefix}', fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.show()

def load_paired_reynolds_multi_3channel_single_file(file_path, lr_dim, hr_dim, reynolds_list=None):
    """
    Loads data from a single HDF5 file.
    """
    xs_lr, xs_hr, used_res, bc_types = [], [], [], []
    
    print(f"📂 Loading from: {file_path}")
    try:
        with h5py.File(file_path, 'r') as f:
            all_keys = list(f.keys())
            re_numbers_in_file = sorted(list(set(int(k.split('_')[0][2:]) for k in all_keys if k.startswith("Re"))))
            
            # Filter if reynolds_list provided
            if reynolds_list:
                re_numbers_in_file = [r for r in re_numbers_in_file if r in reynolds_list]
            
            print(f"   Loading Reynolds numbers: {re_numbers_in_file}")
            
            # Detect BC type
            if all_keys:
                bc_type = f[all_keys[0]].attrs.get('bc_type', 'unknown')
            else:
                bc_type = 'unknown'

            for Re in re_numbers_in_file:
                g_lr = f"Re{Re}_mesh{lr_dim}x{lr_dim}"
                g_hr = f"Re{Re}_mesh{hr_dim}x{hr_dim}"
                
                if g_lr in all_keys and g_hr in all_keys:
                     # Check if all components exist
                    if all(comp in f[g_lr] and comp in f[g_hr] for comp in ['u', 'v', 'p']):
                        # Load raw data
                        lr_u_raw = f[g_lr]['u'][()].astype(np.float32)
                        lr_v_raw = f[g_lr]['v'][()].astype(np.float32)
                        lr_p_raw = f[g_lr]['p'][()].astype(np.float32)
                        
                        hr_u_raw = f[g_hr]['u'][()].astype(np.float32)
                        hr_v_raw = f[g_hr]['v'][()].astype(np.float32)
                        hr_p_raw = f[g_hr]['p'][()].astype(np.float32)
                        
                        # Get physical domain dimensions
                        lx_lr = f[g_lr].attrs.get('lx', 1.0)
                        ly_lr = f[g_lr].attrs.get('ly', 1.0)
                        lx_hr = f[g_hr].attrs.get('lx', 1.0)
                        ly_hr = f[g_hr].attrs.get('ly', 1.0)
                        
                        # Reshape
                        lr_u = lr_u_raw.reshape(lr_dim, lr_dim, order='F')
                        lr_v = lr_v_raw.reshape(lr_dim, lr_dim, order='F')
                        lr_p = lr_p_raw.reshape(lr_dim, lr_dim, order='F')
                        
                        hr_u = hr_u_raw.reshape(hr_dim, hr_dim, order='F')
                        hr_v = hr_v_raw.reshape(hr_dim, hr_dim, order='F')
                        hr_p = hr_p_raw.reshape(hr_dim, hr_dim, order='F')
                        
                        # Aspect ratio correction
                        aspect_ratio_threshold = 0.05
                        if abs(lx_lr - ly_lr) / max(lx_lr, ly_lr) > aspect_ratio_threshold:
                            lr_fields = {'u': lr_u, 'v': lr_v, 'p': lr_p}
                            lr_corrected = reshape_rectangular_to_square(lr_fields, lr_dim, lr_dim, lx_lr, ly_lr)
                            lr_u, lr_v, lr_p = lr_corrected['u'], lr_corrected['v'], lr_corrected['p']
                        
                        if abs(lx_hr - ly_hr) / max(lx_hr, ly_hr) > aspect_ratio_threshold:
                            hr_fields = {'u': hr_u, 'v': hr_v, 'p': hr_p}
                            hr_corrected = reshape_rectangular_to_square(hr_fields, hr_dim, hr_dim, lx_hr, ly_hr)
                            hr_u, hr_v, hr_p = hr_corrected['u'], hr_corrected['v'], hr_corrected['p']
                        
                        # Stack
                        xs_lr.append(np.stack([lr_u, lr_v, lr_p], axis=-1))
                        xs_hr.append(np.stack([hr_u, hr_v, hr_p], axis=-1))
                        used_res.append(Re)
                        bc_types.append(bc_type)

    except Exception as e:
        print(f"⚠️  Error opening file: {e}")
        
    return np.array(xs_lr), np.array(xs_hr), np.array(used_res), np.array(bc_types)

In [ ]:
# =========================
# MODEL ARCHITECTURE
# =========================

def gradient_difference_loss(y_true, y_pred):
    true_dx = y_true[:, :, 1:, :] - y_true[:, :, :-1, :]
    true_dy = y_true[:, 1:, :, :] - y_true[:, :-1, :, :]
    pred_dx = y_pred[:, :, 1:, :] - y_pred[:, :, :-1, :]
    pred_dy = y_pred[:, 1:, :, :] - y_pred[:, :-1, :, :]
    loss_x = tf.reduce_mean(tf.abs(true_dx - pred_dx))
    loss_y = tf.reduce_mean(tf.abs(true_dy - pred_dy))
    return loss_x + loss_y

def spectral_loss(y_true, y_pred):
    y_true_complex = tf.cast(y_true, tf.complex64)
    y_pred_complex = tf.cast(y_pred, tf.complex64)
    fft_true = tf.signal.fft2d(y_true_complex)
    fft_pred = tf.signal.fft2d(y_pred_complex)
    mag_true = tf.abs(fft_true)
    mag_pred = tf.abs(fft_pred)
    return tf.reduce_mean(tf.abs(mag_true - mag_pred))

@tf.keras.utils.register_keras_serializable()
class CompositeLoss(tf.keras.losses.Loss):
    def __init__(self, alpha=1.0, beta=0.1, gamma=0.05, adaptive=False, name='composite_loss'):
        super().__init__(name=name)
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma
        self.adaptive = adaptive
        if adaptive:
            self.l1_ema = tf.Variable(1.0, trainable=False)
            self.grad_ema = tf.Variable(1.0, trainable=False)
            self.spec_ema = tf.Variable(1.0, trainable=False)
            self.ema_decay = 0.9
    
    def call(self, y_true, y_pred):
        l1_loss = tf.reduce_mean(tf.abs(y_true - y_pred))
        grad_loss = gradient_difference_loss(y_true, y_pred)
        spec_loss = spectral_loss(y_true, y_pred)
        if self.adaptive:
            self.l1_ema.assign(self.ema_decay * self.l1_ema + (1 - self.ema_decay) * l1_loss)
            self.grad_ema.assign(self.ema_decay * self.grad_ema + (1 - self.ema_decay) * grad_loss)
            self.spec_ema.assign(self.ema_decay * self.spec_ema + (1 - self.ema_decay) * spec_loss)
            alpha_adaptive = 1.0 / (self.l1_ema + 1e-8)
            beta_adaptive = 1.0 / (self.grad_ema + 1e-8)
            gamma_adaptive = 1.0 / (self.spec_ema + 1e-8)
            total = alpha_adaptive + beta_adaptive + gamma_adaptive
            total_loss = (alpha_adaptive/total) * l1_loss + (beta_adaptive/total) * grad_loss + (gamma_adaptive/total) * spec_loss
        else:
            total_loss = self.alpha * l1_loss + self.beta * grad_loss + self.gamma * spec_loss
        return total_loss

    def get_config(self):
        config = super().get_config()
        config.update({
            "alpha": self.alpha,
            "beta": self.beta,
            "gamma": self.gamma,
            "adaptive": self.adaptive,
        })
        return config

def conv_block(x, filters, kernel_size=3, activation='relu'):
    x = layers.Conv2D(filters, kernel_size, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(activation)(x)
    return x

def build_residual_unet(input_shape, base_filters=32, depth=3, activation='relu'):
    inputs = layers.Input(shape=input_shape, name='unet_input')
    encoder_outputs = []
    x = inputs
    for i in range(depth):
        filters = base_filters * (2 ** i)
        x = conv_block(x, filters, activation=activation)
        x = conv_block(x, filters, activation=activation)
        encoder_outputs.append(x)
        if i < depth - 1:
            x = layers.MaxPooling2D(pool_size=2)(x)
    for i in range(depth - 2, -1, -1):
        filters = base_filters * (2 ** i)
        x = layers.UpSampling2D(size=2, interpolation='bilinear')(x)
        x = layers.Conv2D(filters, 3, padding='same')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation(activation)(x)
        x = layers.Concatenate()([x, encoder_outputs[i]])
        x = conv_block(x, filters, activation=activation)
        x = conv_block(x, filters, activation=activation)
    residual = layers.Conv2D(input_shape[-1], 1, padding='same', name='residual_output')(x)
    return Model(inputs, residual, name='residual_unet')

@tf.keras.utils.register_keras_serializable()
class InterpolateRefineModel(Model):
    def __init__(self, unet_model, target_size, **kwargs):
        super().__init__(**kwargs)
        self.unet = unet_model
        self.target_size = target_size
    
    def call(self, inputs, training=False):
        interpolated = bicubic_interpolate_batch(inputs, self.target_size)
        residual = self.unet(interpolated, training=training)
        residual = residual * 0.1
        refined = interpolated + residual
        return refined

    def get_config(self):
        config = super().get_config()
        config.update({
            "unet_model": self.unet,
            "target_size": self.target_size,
        })
        return config
    
    @classmethod
    def from_config(cls, config):
        unet_model = config.pop("unet_model") # This might be serialized
        # Note: Deserializing nested models in custom objects can be tricky. 
        # Often easier to rebuild or load weights if we have the architecture.
        # For loading .h5 model directly, Keras handles it if registered.
        return cls(unet_model, **config)

In [ ]:
# =========================
# LOAD RESOURCES
# =========================

# 1. Load Normalization Statistics
norm_stats = {}
print(f"📂 Loading stats from: {STATS_FILE}")
if os.path.exists(STATS_FILE):
    with open(STATS_FILE, 'r') as f:
        for line in f:
            if line.startswith("#") or not line.strip(): continue
            parts = line.strip().split()
            if len(parts) == 3:
                norm_stats[parts[0]] = (float(parts[1]), float(parts[2]))
    print("   Stats loaded:", norm_stats)
else:
    print(f"⚠️ Stats file not found at {STATS_FILE}")

# 2. Load Trained Models
trained_models = {}
for dim in STAGE_DIMS:
    model_path = MODEL_FILES.get(dim)
    if model_path and os.path.exists(model_path):
        print(f"📂 Loading model for target dim {dim}: {model_path}")
        try:
            # Load the U-Net (saved as functional model)
            # We use compile=False because we only need it for inference
            unet = tf.keras.models.load_model(model_path, compile=False)
            
            # Wrap in InterpolateRefineModel
            stage_model = InterpolateRefineModel(
                unet_model=unet,
                target_size=(dim, dim),
                name=f"stage_to_{dim}"
            )
            trained_models[dim] = stage_model
            print(f"   ✅ Model loaded successfully.")
        except Exception as e:
            print(f"   ⚠️ Failed to load model: {e}")
    else:
        print(f"   ⚠️ Model file not found: {model_path}")

def run_cascade_inference(x_lr, start_dim=10):
    """
    Run data through the full cascade.
    x_lr: Input data at start_dim (Normalized)
    """
    current_x = x_lr
    current_dim = start_dim
    
    predictions = {}
    
    for target_dim in STAGE_DIMS:
        if target_dim > current_dim:
            if target_dim in trained_models:
                # Predict
                current_x = trained_models[target_dim].predict(current_x, verbose=0)
                predictions[target_dim] = current_x
                current_dim = target_dim
            else:
                print(f"⚠️ Missing model for stage {current_dim}->{target_dim}, stopping cascade.")
                break
                
    return current_x, predictions

In [ ]:
# =========================
# TEST CASE 1: LID DRIVEN CAVITY (LDC)
# =========================

# --------------------------
# CONFIGURATION
# --------------------------
LDC_FILE_PATH = "simulation_result.h5"  # <--- UPDATE THIS PATH to your local file
RE_TO_TEST = [1000]  # Or any other available Reynolds number

# --------------------------
# LOAD AND RUN
# --------------------------
print(f"🚀 Running LDC Evaluation on Re={RE_TO_TEST}...")

if os.path.exists(LDC_FILE_PATH):
    # Load LR input (10x10) and HR target (400x400)
    x_lr, x_hr, res, _ = load_paired_reynolds_multi_3channel_single_file(
        LDC_FILE_PATH, 
        lr_dim=10, 
        hr_dim=400,
        reynolds_list=RE_TO_TEST
    )
    
    if len(x_lr) > 0:
        print(f"   Loaded {len(x_lr)} samples.")
        
        # Normalize
        x_lr_norm = normalize_with_stats(x_lr, norm_stats)
        
        # Run Cascade
        final_pred_norm, intermediate_preds = run_cascade_inference(x_lr_norm, start_dim=10)
        
        # Visualize first sample
        idx = 0
        print(f"   Visualizing sample {idx} (Re={res[idx]})...")
        plot_comparison(
            input_lr=x_lr[idx],      # Raw LR (for visu)
            target_hr=x_hr[idx],     # Raw HR (for visu)
            pred_hr=final_pred_norm[idx], # Model output (Normalized)
            stats=norm_stats,
            title_prefix=f"LDC Test Case - Re={res[idx]}"
        )
        
        # Compute Metrics
        metrics = {}
        for dim, pred in intermediate_preds.items():
            # (Optional) Evaluate intermediate stages if we had HR data for them
            pass
            
        # Final evaluation
        target_real = x_hr
        pred_real = denormalize_with_stats(final_pred_norm, norm_stats)
        
        mae_u = np.mean(np.abs(target_real[..., 0] - pred_real[..., 0]))
        mae_v = np.mean(np.abs(target_real[..., 1] - pred_real[..., 1]))
        mae_p = np.mean(np.abs(target_real[..., 2] - pred_real[..., 2]))
        
        print(f"   📊 Final Metrics (Re={res[idx]}):")
        print(f"      MAE (u): {mae_u:.6f}")
        print(f"      MAE (v): {mae_v:.6f}")
        print(f"      MAE (p): {mae_p:.6f}")
        print(f"      MAE (Avg): {(mae_u+mae_v+mae_p)/3:.6f}")
        
    else:
        print("   ⚠️ No samples found for the specified Reynolds numbers.")
else:
    print(f"⚠️ LDC file not found: {LDC_FILE_PATH}")

In [ ]:
# =========================
# TEST CASE 2: BACKWARD FACING STEP (BFS)
# =========================

# --------------------------
# CONFIGURATION
# --------------------------
BFS_FILE_PATH = "simulation_result_bfs.h5"   # <--- UPDATE THIS PATH
BFS_RE_TO_TEST = [100]  # Usually Re=100 for BFS in this dataset

# --------------------------
# LOAD AND RUN
# --------------------------
print(f"🚀 Running BFS Evaluation on Re={BFS_RE_TO_TEST}...")

if os.path.exists(BFS_FILE_PATH):
    # Load LR input (10x10) and HR target (400x400)
    x_bfs_lr, x_bfs_hr, bfs_res, _ = load_paired_reynolds_multi_3channel_single_file(
        BFS_FILE_PATH, 
        lr_dim=10, 
        hr_dim=400,
        reynolds_list=BFS_RE_TO_TEST
    )
    
    if len(x_bfs_lr) > 0:
        print(f"   Loaded {len(x_bfs_lr)} BFS samples.")
        
        # Normalize
        x_bfs_lr_norm = normalize_with_stats(x_bfs_lr, norm_stats)
        
        # Run Cascade
        bfs_pred_norm, bfs_intermediate = run_cascade_inference(x_bfs_lr_norm, start_dim=10)
        
        # Visualize first sample
        idx = 0
        print(f"   Visualizing BFS sample {idx} (Re={bfs_res[idx]})...")
        plot_comparison(
            input_lr=x_bfs_lr[idx],
            target_hr=x_bfs_hr[idx],
            pred_hr=bfs_pred_norm[idx],
            stats=norm_stats,
            title_prefix=f"BFS Test Case - Re={bfs_res[idx]}"
        )
        
        # Compute Metrics
        target_real = x_bfs_hr
        pred_real = denormalize_with_stats(bfs_pred_norm, norm_stats)
        
        mae_u = np.mean(np.abs(target_real[..., 0] - pred_real[..., 0]))
        mae_v = np.mean(np.abs(target_real[..., 1] - pred_real[..., 1]))
        mae_p = np.mean(np.abs(target_real[..., 2] - pred_real[..., 2]))
        
        print(f"   📊 BFS Final Metrics (Re={bfs_res[idx]}):")
        print(f"      MAE (u): {mae_u:.6f}")
        print(f"      MAE (v): {mae_v:.6f}")
        print(f"      MAE (p): {mae_p:.6f}")
        print(f"      MAE (Avg): {(mae_u+mae_v+mae_p)/3:.6f}")

    else:
        print("   ⚠️ No BFS samples found for the specified Reynolds numbers.")
else:
    print(f"⚠️ BFS file not found: {BFS_FILE_PATH}")